In [1]:
!pip install plotly ipywidgets pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.9 MB/s eta 0:00:00


In [2]:
from google.colab import output
output.enable_custom_widget_manager()

In [3]:
# ============================================================
# DASHBOARD PROFESIONAL EMG MANO - VERSION 2
# Simulación biomédica interactiva para Google Colab
# ============================================================

# Si hiciera falta en Colab:
# !pip install plotly ipywidgets pandas numpy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import os

np.random.seed(123)

# ============================================================
# CONFIGURACION GENERAL
# ============================================================

MUSCLES = [
    "Flexor Digitorum",
    "Extensor Digitorum",
    "Thenar",
    "Interossei"
]

GESTURES = {
    "Reposo":              [0.05, 0.05, 0.04, 0.03],
    "Cerrar mano":         [0.95, 0.25, 0.35, 0.65],
    "Abrir mano":          [0.20, 0.95, 0.20, 0.35],
    "Pinza fina":          [0.45, 0.15, 0.95, 0.85],
    "Agarre fuerte":       [1.00, 0.35, 0.55, 0.75],
    "Señalar":             [0.35, 0.20, 0.40, 0.92]
}

# ============================================================
# FUNCIONES DE APOYO
# ============================================================

def moving_average(x, w):
    w = max(1, int(w))
    return np.convolve(x, np.ones(w) / w, mode="same")

def band_limited_noise(n, fs, f_low=20, f_high=180):
    freqs = np.fft.rfftfreq(n, d=1/fs)
    spectrum = np.zeros(len(freqs), dtype=complex)

    band = (freqs >= f_low) & (freqs <= f_high)
    phases = np.exp(1j * 2 * np.pi * np.random.rand(np.sum(band)))
    amps = np.random.rand(np.sum(band))
    spectrum[band] = amps * phases

    sig = np.fft.irfft(spectrum, n=n)
    sig = sig / (np.std(sig) + 1e-12)
    return sig

def emg_burst_envelope(t, activation, burst_rate=0.6):
    base = 0.08 + 0.9 * activation
    mod = (
        0.10 * np.sin(2 * np.pi * burst_rate * t) +
        0.05 * np.sin(2 * np.pi * (burst_rate * 2.2) * t + 0.6)
    )
    env = np.clip(base + mod, 0.02, None)
    return env

def simulate_emg_channel(fs, duration, activation=0.5, fatigue=0.2, noise=0.05, tremor=0.02):
    n = int(fs * duration)
    t = np.arange(n) / fs

    # La fatiga desplaza energía a frecuencias menores
    f_high = max(70, 180 - int(100 * fatigue))
    raw_emg = band_limited_noise(n, fs, 20, f_high)

    envelope = emg_burst_envelope(t, activation)

    fatigue_gain = 1.0 + 0.5 * fatigue
    tremor_component = tremor * 0.25 * np.sin(2 * np.pi * 10 * t)
    line_noise = noise * 0.08 * np.sin(2 * np.pi * 50 * t)
    white_noise = noise * 0.25 * np.random.randn(n)

    signal = raw_emg * envelope * fatigue_gain
    signal = signal + tremor_component + line_noise + white_noise

    signal = signal / (np.max(np.abs(signal)) + 1e-12)
    signal = signal * (0.15 + 1.8 * activation)

    return t, signal

def generate_hand_emg(fs, duration, gesture, contraction, fatigue, noise, tremor):
    pattern = np.array(GESTURES[gesture], dtype=float)
    pattern = np.clip(pattern * contraction, 0.01, 1.0)

    df = pd.DataFrame()
    t_global = None

    for i, muscle in enumerate(MUSCLES):
        local_fatigue = np.clip(fatigue * (0.9 + 0.2 * np.random.rand()), 0, 1)
        local_noise = np.clip(noise * (0.8 + 0.4 * np.random.rand()), 0, 1)
        local_tremor = np.clip(tremor * (0.8 + 0.4 * np.random.rand()), 0, 1)

        t, sig = simulate_emg_channel(
            fs=fs,
            duration=duration,
            activation=pattern[i],
            fatigue=local_fatigue,
            noise=local_noise,
            tremor=local_tremor
        )
        df[muscle] = sig
        t_global = t

    df["Time"] = t_global
    return df, pattern

# ============================================================
# FEATURES EMG
# ============================================================

def zero_crossings(x):
    return np.sum(np.diff(np.signbit(x)) > 0)

def waveform_length(x):
    return np.sum(np.abs(np.diff(x)))

def slope_sign_changes(x, threshold=0.01):
    dx = np.diff(x)
    return np.sum(
        ((dx[:-1] * dx[1:]) < 0) &
        (np.abs(dx[:-1]) > threshold) &
        (np.abs(dx[1:]) > threshold)
    )

def emg_features(x, fs):
    rms = np.sqrt(np.mean(x**2))
    mav = np.mean(np.abs(x))
    zc = zero_crossings(x)
    wl = waveform_length(x)
    ssc = slope_sign_changes(x)

    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    psd = np.abs(np.fft.rfft(x))**2
    psd_sum = np.sum(psd) + 1e-12

    mean_freq = np.sum(freqs * psd) / psd_sum
    cumulative = np.cumsum(psd)
    median_freq = freqs[np.searchsorted(cumulative, cumulative[-1] / 2)]

    return {
        "RMS": rms,
        "MAV": mav,
        "ZC": zc,
        "WL": wl,
        "SSC": ssc,
        "Fmean": mean_freq,
        "Fmed": median_freq
    }

def compute_feature_table(df, fs):
    rows = []
    for muscle in MUSCLES:
        feat = emg_features(df[muscle].values, fs)
        feat["Músculo"] = muscle
        rows.append(feat)
    table = pd.DataFrame(rows)
    return table[["Músculo", "RMS", "MAV", "ZC", "WL", "SSC", "Fmean", "Fmed"]]

# ============================================================
# CLASIFICADOR SIMPLE DE GESTO
# ============================================================

def classify_gesture(activation_pattern):
    best_gesture = None
    best_score = 1e9

    for gesture_name, ref in GESTURES.items():
        ref = np.array(ref, dtype=float)
        ref = ref / (np.max(ref) + 1e-12)
        current = activation_pattern / (np.max(activation_pattern) + 1e-12)
        score = np.linalg.norm(current - ref)

        if score < best_score:
            best_score = score
            best_gesture = gesture_name

    confidence = max(0, 100 - best_score * 100)
    confidence = min(99.9, confidence)
    return best_gesture, confidence

def fatigue_index(feature_table):
    fmed_mean = feature_table["Fmed"].mean()
    rms_mean = feature_table["RMS"].mean()

    score = (1.8 * rms_mean) + (180 - fmed_mean) / 180
    score = float(np.clip(score * 35, 0, 100))
    return score

def fatigue_label(score):
    if score < 35:
        return "Baja"
    elif score < 65:
        return "Moderada"
    else:
        return "Alta"

# ============================================================
# DASHBOARD
# ============================================================

def build_dashboard(df, fs, activation_pattern, true_gesture):
    t = df["Time"].values
    feature_table = compute_feature_table(df, fs)

    pred_gesture, confidence = classify_gesture(activation_pattern)
    fat_score = fatigue_index(feature_table)
    fat_text = fatigue_label(fat_score)

    fig = make_subplots(
        rows=4, cols=2,
        subplot_titles=(
            "Señales EMG", "Mapa de activación muscular",
            "Envolvente EMG", "Espectro en frecuencia",
            "RMS por canal", "Frecuencia mediana",
            "Clasificación del gesto", "Índice de fatiga"
        ),
        specs=[
            [{"type": "xy"}, {"type": "heatmap"}],
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "indicator"}, {"type": "indicator"}]
        ],
        vertical_spacing=0.08,
        horizontal_spacing=0.10
    )

    # Señales
    for muscle in MUSCLES:
        fig.add_trace(
            go.Scatter(
                x=t, y=df[muscle],
                mode="lines",
                name=muscle,
                line=dict(width=1.4)
            ),
            row=1, col=1
        )

    # Heatmap de activación
    fig.add_trace(
        go.Heatmap(
            z=[activation_pattern],
            x=MUSCLES,
            y=["Activación"],
            text=[[f"{v:.2f}" for v in activation_pattern]],
            texttemplate="%{text}",
            showscale=True
        ),
        row=1, col=2
    )

    # Envolvente
    window = max(5, int(fs * 0.03))
    for muscle in MUSCLES:
        env = moving_average(np.abs(df[muscle].values), window)
        fig.add_trace(
            go.Scatter(
                x=t, y=env,
                mode="lines",
                showlegend=False,
                line=dict(width=2)
            ),
            row=2, col=1
        )

    # FFT
    for muscle in MUSCLES:
        x = df[muscle].values
        freqs = np.fft.rfftfreq(len(x), d=1/fs)
        mag = np.abs(np.fft.rfft(x))
        fig.add_trace(
            go.Scatter(
                x=freqs,
                y=mag,
                mode="lines",
                showlegend=False,
                line=dict(width=1.5)
            ),
            row=2, col=2
        )

    # RMS
    fig.add_trace(
        go.Bar(
            x=feature_table["Músculo"],
            y=feature_table["RMS"],
            text=[f"{v:.3f}" for v in feature_table["RMS"]],
            textposition="outside",
            showlegend=False
        ),
        row=3, col=1
    )

    # Fmed
    fig.add_trace(
        go.Bar(
            x=feature_table["Músculo"],
            y=feature_table["Fmed"],
            text=[f"{v:.1f} Hz" for v in feature_table["Fmed"]],
            textposition="outside",
            showlegend=False
        ),
        row=3, col=2
    )

    # Indicador clasificación
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=confidence,
            title={"text": f"Predicción: {pred_gesture}<br><span style='font-size:12px'>Real: {true_gesture}</span>"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"thickness": 0.3},
                "steps": [
                    {"range": [0, 40], "color": "#f8d7da"},
                    {"range": [40, 75], "color": "#fff3cd"},
                    {"range": [75, 100], "color": "#d1e7dd"}
                ]
            }
        ),
        row=4, col=1
    )

    # Indicador fatiga
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=fat_score,
            title={"text": f"Fatiga: {fat_text}"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"thickness": 0.3},
                "steps": [
                    {"range": [0, 35], "color": "#d1e7dd"},
                    {"range": [35, 65], "color": "#fff3cd"},
                    {"range": [65, 100], "color": "#f8d7da"}
                ]
            }
        ),
        row=4, col=2
    )

    fig.update_layout(
        title="Dashboard Biomédico Interactivo - Señales Mioeléctricas de la Mano",
        height=1400,
        width=1500,
        template="plotly_white",
        margin=dict(l=40, r=40, t=80, b=40)
    )

    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=1)
    fig.update_yaxes(title_text="Amplitud [mV simulados]", row=1, col=1)

    fig.update_xaxes(title_text="Tiempo [s]", row=2, col=1)
    fig.update_yaxes(title_text="Envolvente", row=2, col=1)

    fig.update_xaxes(title_text="Frecuencia [Hz]", range=[0, 250], row=2, col=2)
    fig.update_yaxes(title_text="Magnitud", row=2, col=2)

    fig.update_xaxes(title_text="Músculo", row=3, col=1)
    fig.update_yaxes(title_text="RMS", row=3, col=1)

    fig.update_xaxes(title_text="Músculo", row=3, col=2)
    fig.update_yaxes(title_text="Hz", row=3, col=2)

    return fig, feature_table, pred_gesture, confidence, fat_score, fat_text

# ============================================================
# EXPORTACION
# ============================================================

LAST = {"df": None, "features": None, "fig": None}

def export_csv():
    if LAST["df"] is None or LAST["features"] is None:
        print("Primero generá la simulación.")
        return

    LAST["df"].to_csv("emg_hand_signals.csv", index=False)
    LAST["features"].to_csv("emg_hand_features.csv", index=False)
    print("Exportados: emg_hand_signals.csv y emg_hand_features.csv")

def export_html():
    if LAST["fig"] is None:
        print("Primero generá la simulación.")
        return

    LAST["fig"].write_html("dashboard_emg_mano.html")
    print("Exportado: dashboard_emg_mano.html")

# ============================================================
# WIDGETS
# ============================================================

gesture_widget = widgets.Dropdown(
    options=list(GESTURES.keys()),
    value="Cerrar mano",
    description="Gesto:",
    layout=widgets.Layout(width="320px")
)

contraction_widget = widgets.FloatSlider(
    value=0.85, min=0.10, max=1.20, step=0.05,
    description="Contracción:",
    readout_format=".2f",
    layout=widgets.Layout(width="430px")
)

fatigue_widget = widgets.FloatSlider(
    value=0.20, min=0.00, max=1.00, step=0.05,
    description="Fatiga:",
    readout_format=".2f",
    layout=widgets.Layout(width="430px")
)

noise_widget = widgets.FloatSlider(
    value=0.08, min=0.00, max=0.50, step=0.01,
    description="Ruido:",
    readout_format=".2f",
    layout=widgets.Layout(width="430px")
)

tremor_widget = widgets.FloatSlider(
    value=0.05, min=0.00, max=1.00, step=0.05,
    description="Temblor:",
    readout_format=".2f",
    layout=widgets.Layout(width="430px")
)

duration_widget = widgets.FloatSlider(
    value=6.0, min=2.0, max=12.0, step=0.5,
    description="Duración [s]:",
    readout_format=".1f",
    layout=widgets.Layout(width="430px")
)

fs_widget = widgets.IntSlider(
    value=1000, min=500, max=4000, step=100,
    description="Fs [Hz]:",
    layout=widgets.Layout(width="430px")
)

btn_csv = widgets.Button(
    description="Exportar CSV",
    button_style="success",
    layout=widgets.Layout(width="180px")
)

btn_html = widgets.Button(
    description="Exportar HTML",
    button_style="info",
    layout=widgets.Layout(width="180px")
)

output = widgets.Output()

# ============================================================
# ACTUALIZACION
# ============================================================

def refresh_dashboard(*args):
    with output:
        clear_output(wait=True)

        fs = fs_widget.value
        duration = duration_widget.value
        gesture = gesture_widget.value
        contraction = contraction_widget.value
        fatigue = fatigue_widget.value
        noise = noise_widget.value
        tremor = tremor_widget.value

        df, activation_pattern = generate_hand_emg(
            fs=fs,
            duration=duration,
            gesture=gesture,
            contraction=contraction,
            fatigue=fatigue,
            noise=noise,
            tremor=tremor
        )

        fig, feature_table, pred_gesture, confidence, fat_score, fat_text = build_dashboard(
            df=df,
            fs=fs,
            activation_pattern=activation_pattern,
            true_gesture=gesture
        )

        LAST["df"] = df
        LAST["features"] = feature_table
        LAST["fig"] = fig

        display(fig)

        print("Resumen biomédico")
        print(f"Gesto seleccionado: {gesture}")
        print(f"Gesto clasificado: {pred_gesture}")
        print(f"Confianza estimada: {confidence:.1f}%")
        print(f"Índice de fatiga: {fat_score:.1f}/100 ({fat_text})")
        print("")

        display(
            feature_table.style.format({
                "RMS": "{:.4f}",
                "MAV": "{:.4f}",
                "ZC": "{:.0f}",
                "WL": "{:.2f}",
                "SSC": "{:.0f}",
                "Fmean": "{:.2f}",
                "Fmed": "{:.2f}"
            })
        )

def on_csv_clicked(b):
    with output:
        export_csv()

def on_html_clicked(b):
    with output:
        export_html()

btn_csv.on_click(on_csv_clicked)
btn_html.on_click(on_html_clicked)

gesture_widget.observe(refresh_dashboard, names="value")
contraction_widget.observe(refresh_dashboard, names="value")
fatigue_widget.observe(refresh_dashboard, names="value")
noise_widget.observe(refresh_dashboard, names="value")
tremor_widget.observe(refresh_dashboard, names="value")
duration_widget.observe(refresh_dashboard, names="value")
fs_widget.observe(refresh_dashboard, names="value")

# ============================================================
# INTERFAZ
# ============================================================

left_controls = widgets.VBox([
    gesture_widget,
    contraction_widget,
    fatigue_widget,
    noise_widget
])

right_controls = widgets.VBox([
    tremor_widget,
    duration_widget,
    fs_widget,
    widgets.HBox([btn_csv, btn_html])
])

ui = widgets.HBox([left_controls, right_controls])

display(HTML("<h2>Simulador Profesional de Señales Mioeléctricas de la Mano</h2>"))
display(HTML("<p>Modelo didáctico de EMG superficial multicanal orientado a docencia, prototipado biomédico y análisis de gestos.</p>"))
display(ui)
display(output)

refresh_dashboard()

Output()